In [ ]:
import numpy as np
from qutip import *
from scipy.linalg import sqrtm, eigvalsh
from scipy.stats import linregress
from numba import njit, prange
import pickle
import os
import time

In [ ]:
%matplotlib ipympl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import Image, display, Math

## Fidelity
Generic definition : 
$$ \mathcal{F}\left( \rho, \sigma \right) = \left( Tr \left[ \sqrt{ \sqrt{\rho} \sigma \sqrt{\rho} }\right] \right)^{2} $$ 
Definition for $ \rho $ Pure State and $ \sigma $ Mixed State : 
$$ \mathcal{F}\left( \rho, \sigma \right) = \langle \psi_{\sigma} | \sigma | \psi_{\sigma} \rangle $$
Definition for Pure State : 
$$ \mathcal{F}\left( \rho, \sigma \right) = |\langle \psi_{\rho} | \psi_{\sigma} \rangle|^{2} $$
Definition for Qubits : 
$$ \mathcal{F}\left( \rho, \sigma \right) = Tr\left( \rho \, \sigma \right) + 2 \sqrt{Det\left ( \rho \right) \, Det\left ( \sigma \right)} $$
## Trace Distance
Generic definition : 
$$ \mathcal{T}\left( \rho, \sigma \right) = \frac{1}{2} Tr \left[ \sqrt{\left( \rho - \sigma \right)^{\dagger} \left( \rho - \sigma  \right)} \right] $$
### Relationship : Fuchs-van de Graaf inequality
$$ 1 - \sqrt{\mathcal{F}\left( \rho, \sigma \right)} \leq \mathcal{T}\left( \rho, \sigma \right) \leq \sqrt{1 - \mathcal{F}\left( \rho, \sigma \right)} $$

In [ ]:
def fidelity_generic(rho, sigma):
    """
    Calculate the quantum fidelity between two generic density matrices.
    Formula: F(rho, sigma) = ( Tr[ sqrt( sqrt(rho) * sigma * sqrt(rho) ) ] )^2
    
    This version avoids scipy.linalg.sqrtm to prevent RuntimeWarnings, 
    using stable eigenvalue decomposition instead.
    
    Parameters:
        rho (numpy.ndarray): First density matrix (NxN).
        sigma (numpy.ndarray): Second density matrix (NxN).
        
    Returns:
        float: The fidelity between rho and sigma (real number between 0 and 1).
    """
    # 1. Square root of rho using eigenvalue decomposition
    evals_rho, evecs_rho = np.linalg.eigh(rho)
    # Truncate any negative noise to 0.0 before taking the square root
    evals_rho = np.maximum(evals_rho, 0.0) 
    sqrt_rho = evecs_rho @ np.diag(np.sqrt(evals_rho)) @ evecs_rho.conj().T
    
    # 2. Inner matrix: sqrt(rho) * sigma * sqrt(rho)
    inner_matrix = sqrt_rho @ sigma @ sqrt_rho
    
    # Force exact Hermiticity to remove any small imaginary noise
    inner_matrix = 0.5 * (inner_matrix + inner_matrix.conj().T)
    
    # 3. Trace of the square root is the sum of the square roots of the eigenvalues
    evals_inner = eigvalsh(inner_matrix)
    # Again, truncate negative noise to 0.0 before square root
    evals_inner = np.maximum(evals_inner, 0.0)
    
    fidelity = np.sum(np.sqrt(evals_inner))**2
    
    # Ensure numerical errors do not push fidelity slightly above 1.0
    return min(1.0, fidelity)
    

In [ ]:
@njit
def fidelity_qubit(rho, sigma):
    """
    Calculate the exact quantum fidelity between two single-qubit (2x2) density matrices.
    Formula: F(rho, sigma) = Tr(rho * sigma) + 2 * sqrt(Det(rho) * Det(sigma))
    """
    # Trace of the matrix product
    tr_term = np.real(np.trace(rho @ sigma))
    
    # Determinants of the two density matrices
    det_rho = np.real(np.linalg.det(rho))
    det_sigma = np.real(np.linalg.det(sigma))
    
    # FIX NUMERICO: Tronchiamo a 0 gli eventuali valori negativi infinitesimi
    det_rho = max(0.0, det_rho)
    det_sigma = max(0.0, det_sigma)
    
    # Calculate fidelity using the analytical formula for qubits
    fidelity = tr_term + 2.0 * np.sqrt(det_rho * det_sigma)
    
    return fidelity

In [ ]:
def trace_distance_generic(rho, sigma):
    """
    Calculate the Trace Distance between two generic density matrices.
    Formula: T(rho, sigma) = 1/2 * Tr[ sqrt( (rho - sigma)^dagger * (rho - sigma) ) ]
    
    Parameters:
        rho (numpy.ndarray): First density matrix (NxN).
        sigma (numpy.ndarray): Second density matrix (NxN).
        
    Returns:
        float: The trace distance between rho and sigma (real number between 0 and 1).
    """
    # Difference between the matrices
    diff = rho - sigma
    
    # Force exact Hermiticity to remove numerical noise
    diff = 0.5 * (diff + diff.conj().T)
    
    # Calculate the eigenvalues of the strictly Hermitian matrix 'diff'
    eigenvalues = eigvalsh(diff)
    
    # Trace distance is half the sum of the absolute eigenvalues
    t_dist = 0.5 * np.sum(np.abs(eigenvalues))
    
    # Ensure it stays within physical bounds
    return min(1.0, t_dist)
    

In [ ]:
def trace_distance_qubit(rho, sigma):
    """
    Calculate the exact Trace Distance between two single-qubit (2x2) density matrices.
    For a 2x2 traceless Hermitian matrix (rho - sigma), Det(diff) = -lambda^2 <= 0.
    Therefore, the Trace Distance simplifies to sqrt(-Det(rho - sigma)).
    
    Parameters:
        rho (numpy.ndarray): First density matrix (2x2).
        sigma (numpy.ndarray): Second density matrix (2x2).
        
    Returns:
        float: The trace distance between rho and sigma.
    """
    # Difference between the matrices
    diff = rho - sigma
    
    # Determinant of the difference
    det_diff = np.real(np.linalg.det(diff))
    
    # Since det_diff should be <= 0, -det_diff should be >= 0.
    # We use max(0.0, ...) to truncate any negative noise before applying sqrt.
    val_under_sqrt = max(0.0, -det_diff)
    
    t_dist = np.sqrt(val_under_sqrt)
    
    return min(1.0, t_dist)
    

In [ ]:
@njit
def fidelity_qubit_single_term(p00, p11, c10, c01, sigma):
    """
    Build the density matrix from its elements and then
    Calculate the exact quantum fidelity between two single-qubit (2x2) density matrices.
    Formula: F(rho, sigma) = Tr(rho * sigma) + 2 * sqrt(Det(rho) * Det(sigma))
    """
    # 1. Density Matrix build up 
    rho = np.zeros((2, 2), dtype=np.complex128)
    rho[0,0] = p00
    rho[0,1] = c01
    rho[1,0] = c10
    rho[1,1] = p11

    # 2. Trace of the matrix product (Calculated explicitly for 2x2 to avoid Numba issues)
    prod = rho @ sigma
    tr_term = prod[0,0].real + prod[1,1].real
    
    # 3. Determinants of the two 2x2 density matrices (ad - bc)
    det_rho = (rho[0,0] * rho[1,1] - rho[0,1] * rho[1,0]).real
    det_sigma = (sigma[0,0] * sigma[1,1] - sigma[0,1] * sigma[1,0]).real
    
    # 4. Numerical FIX: Truncate to 0 any infinitesimal negative values
    det_rho = max(0.0, det_rho)
    det_sigma = max(0.0, det_sigma)
    
    # 5. Calculate fidelity using the analytical formula for qubits
    fidelity = tr_term + 2.0 * np.sqrt(det_rho * det_sigma)
    
    return fidelity

In [ ]:
@njit
def trace_distance_qubit_single_term(p00, p11, c10, c01, sigma):
    """
    Build the density matrix from its elements and then
    Calculate the exact Trace Distance between two single-qubit (2x2) density matrices.
    For a 2x2 traceless Hermitian matrix (rho - sigma), Det(diff) = -lambda^2 <= 0.
    Therefore, the Trace Distance simplifies to sqrt(-Det(rho - sigma)).
    
    Parameters:
        rho (numpy.ndarray): First density matrix (2x2).
        sigma (numpy.ndarray): Second density matrix (2x2).
        
    Returns:
        float: The trace distance between rho and sigma.
    """
    # Density Matrix build up 
    rho = np.zeros((2, 2), dtype=np.complex128)
    rho[0,0] = p00
    rho[0,1] = c01
    rho[1,0] = c10
    rho[1,1] = p11
    
    # Difference between the matrices
    diff = rho - sigma
    
    # Determinant of the difference
    det_diff = (diff[0,0]*diff[1,1] - diff[0,1]*diff[1,0]).real
    
    # Since det_diff should be <= 0, -det_diff should be >= 0.
    # We use max(0.0, ...) to truncate any negative noise before applying sqrt.
    val_under_sqrt = max(0.0, -det_diff)
    
    t_dist = np.sqrt(val_under_sqrt)
    
    return min(1.0, t_dist)

In [ ]:
@njit
def compute_metrics_over_time(pop_10, pop_01, coh_1001, coh_0110, rho_lindblad_array):
    """
    Computes fidelity and trace distance over all time steps using Numba in C-speed.
    """
    time_steps = len(pop_10)
    
    fidelity_arr = np.zeros(time_steps)
    trace_dist_arr = np.zeros(time_steps)
    
    for t in range(time_steps):
        fidelity_arr[t] = fidelity_qubit_single_term(
            pop_10[t], pop_01[t], coh_1001[t], coh_0110[t], rho_lindblad_array[t]
        )
        trace_dist_arr[t] = trace_distance_qubit_single_term(
            pop_10[t], pop_01[t], coh_1001[t], coh_0110[t], rho_lindblad_array[t]
        )
        
    return fidelity_arr, trace_dist_arr

In [ ]:
# =====================================================================
# NUMBA OPTIMIZED LOOP FOR ALL INDIVIDUAL TRAJECTORIES (2D ARRAYS)
# =====================================================================
@njit(parallel=True)
def compute_metrics_all_trajectories(pop_10, coh_1001, coh_0110, pop_01, rho_lindblad_array):
    """
    Computes fidelity and trace distance for ALL individual trajectories.
    Inputs are 2D arrays: (time_steps, N_traj) and Lindblad 3D array.
    Uses parallel processing (prange) for maximum speed.
    """
    time_steps = pop_10.shape[0]
    N_traj = pop_10.shape[1]
    
    # Pre-allocate output 2D arrays (time_steps, N_traj)
    fidelity_matrix = np.zeros((time_steps, N_traj))
    trace_dist_matrix = np.zeros((time_steps, N_traj))
    
    # Outer loop over time
    for t in range(time_steps):
        # Parallel loop over all trajectories (this uses all your CPU cores!)
        for n in prange(N_traj):
            fidelity_matrix[t, n] = fidelity_qubit_single_term(
                pop_10[t, n], pop_01[t, n], coh_1001[t, n], coh_0110[t, n], rho_lindblad_array[t]
            )
            trace_dist_matrix[t, n] = trace_distance_qubit_single_term(
                pop_10[t, n], pop_01[t, n], coh_1001[t, n], coh_0110[t, n], rho_lindblad_array[t]
            )
            
    return fidelity_matrix, trace_dist_matrix

## General Setup

In [ ]:
# ====================================
# Physical & Simulation Parameters
# ====================================

# Time parameters
dt = 0.01
tf = 100.0
time_steps = int(tf / dt)

# List of Number of trajectories to analyze
N_traj_list = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 20000]        

In [ ]:
# ============================================================
# GLOBAL CONFIGURATION :    
#   'QJ' → Quantum Jump
#   'SD' → State Diffusion
# ============================================================


MODE = 'SD'   # Switch this to 'SD' or 'QJ'

# Configuration mapping based on MODE
_cfg = {
    'QJ': {
        'Input_file': "../Results/Data/Complete_rho/pop_1.0_result_mode_QJ_dt0p010000_Ntraj20000.npz",
        'Output_dir': "../Results/Plot/Fidelity/Complete/QJ" 
    },
    'SD': {
        'Input_file': "../Results/Data/Complete_rho/pop_1.0_result_mode_SD_dt0p010000_Ntraj20000.npz",
        'Output_dir': "../Results/Plot/Fidelity/Complete/SD"
    },
}

# Apply the selected configuration
cfg = _cfg[MODE]

# Set the global Input and Output_dir dynamically based on the current mode
Input_file = cfg['Input_file']
Output_dir = cfg['Output_dir']

# Create the output directory if it doesn't exist
os.makedirs(Output_dir, exist_ok=True)

print(f"--- Configuration Setup ---")
print(f"Current mode : {MODE}")
print(f"Input Data   : {cfg['Input_file']}")
print(f"Output Plots : {Output_dir}")

In [ ]:
# ===========================
# General Setup for Plotting
# ===========================

# Global Style Settings (Matplotlib rcParams)
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': ':',
    'figure.autolayout': True # plt.tight_layout()
})

# Automatic Figure Saving Function
def save_fig(fig, filename):
    """
    Saves the figure in both PNG or PDF
    """
    path_png = os.path.join(Output_dir, f"{filename}.png")
    # path_pdf = os.path.join(Output_dir, f"{filename}.pdf")  # save in pdf
    
    fig.savefig(path_png, dpi=300, bbox_inches='tight')
    # fig.savefig(path_pdf, bbox_inches='tight') # save in pdf
    print(f"Figure saved in: {Output_dir}/{filename}")

 ### Data Extraction

In [ ]:
# ------------
# Load Results
# ------------
data = np.load(Input_file)

times = data['times']
time_steps = len(times)

# ------------
# Extract data
# ------------

# Extract Lindblad populations
rho_lindblad = data['rho_list_lindblad']
# lindblad_00 = np.real(rho_lindblad[:, 0, 0])
# lindblad_11 = np.real(rho_lindblad[:, 1, 1])
# lindblad_01 = rho_lindblad[:, 0, 1]

# Extract Trajectories data 
pop_00 = data['pop_00']
pop_11 = data['pop_11']
coh_01 = data['coh_01']
coh_10 = np.conj(coh_01)

print("--- Data Loading Completed ---")

# ==============================
# Density Matrix reconstruction
# ==============================

N_traj_tot = pop_00.shape[1]

rho_complete = np.zeros((time_steps, N_traj_tot, 2, 2), dtype=np.complex128)
rho_complete[:, :, 0, 0] = pop_00
rho_complete[:, :, 1, 1] = pop_11
rho_complete[:, :, 0, 1] = coh_01
rho_complete[:, :, 1, 0] = np.conj(coh_01)


## Purity

$ P = Tr[\rho^2]  $

In [ ]:
# =============================================================
# Purity Check via simplified formula: p10^2 + p01^2 + 2*|c|^2 
# =============================================================

print("Starting Purity Check")
start_time = time.time()

n_times, N_traj_tot = pop_00.shape
max_deviation = 0.0

# Chunking to reduce RAM usage
CHUNK_SIZE = 5000
n_chunks = int(np.ceil(N_traj_tot / CHUNK_SIZE))

for i in range(n_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, N_traj_tot)
    
    p00_chunk = pop_00[:, start_idx:end_idx]
    p11_chunk = pop_11[:, start_idx:end_idx]
    c_chunk = coh_01[:, start_idx:end_idx]
    
    # Purity p10^2 + p01^2 + 2*|c|^2
    purity_chunk = p00_chunk**2 + p11_chunk**2 + 2 * (np.abs(c_chunk)**2)
    
    # Max deviation from theroretical value 1.0
    chunk_max_dev = np.max(np.abs(1.0 - purity_chunk))
    
    if chunk_max_dev > max_deviation:
        max_deviation = chunk_max_dev

print(f"Check completed in {time.time() - start_time:.4f} seconds.")
print(f"Maximum deviation from ideal purity (1.0): {max_deviation:.4e}")

# Automatic evaluation of the result
if max_deviation < 1e-10:
    print("\n✅ Test Passed: all trajectoreis are always perfectly pure")
else:
    print("\n⚠️ Warning: Found a significant deviation from purity. Check the normalization in the simulator.")

## Fidelity calculation

In [ ]:
# ==========================================================================
# COMPLETE FIDELITY & TRACE DISTANCE CALCULATION (ALL TRAJECTORIES)
# ==========================================================================

print("Starting complete calculation for all trajectories...")

# Call the parallel Numba function for 2D arrays
all_fidelity_matrix, all_trace_dist_matrix = compute_metrics_all_trajectories(
    pop_00, coh_10, coh_01, pop_11, rho_lindblad
)

print(f"Calculation finished for all trajectories. Shape of fidelity array: {all_fidelity_matrix.shape}")
    

## Plot 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# Fidelity for Avg Traj vs Lindblad in time
# ==========================================

plt.close('all')

# List of trajectories to average over
N_list = [100, 1000, 10000]  

# Create the figure and axis
fig01, ax = plt.subplots(figsize=(8, 5))

for N in N_list:
    # 1. Average the 4 density matrix components over N trajectories
    pop_00_m = np.mean(pop_00[:, :N], axis=1)
    pop_11_m = np.mean(pop_11[:, :N], axis=1)
    coh_10_m = np.mean(coh_10[:, :N], axis=1)
    coh_01_m = np.mean(coh_01[:, :N], axis=1)

    # 2. Calculate fidelity using the optimized Numba function
    # Remember: 3rd arg is coh_10, 4th arg is coh_01 
    fidelity_list, _ = compute_metrics_over_time(
        pop_00_m, pop_11_m, coh_10_m, coh_01_m, rho_lindblad
    )

    # 3. Plot the current N curve
    ax.plot(times, fidelity_list, label=f'N_traj = {N}', alpha=1.0, linewidth=1.5)

# Apply labels and title 
ax.set_xlabel("Time")
ax.set_ylabel(r"Fidelity $\mathcal{F}(\langle\rho\rangle_t, \rho_L)$")
ax.set_title(f"Evolution of Avg Fidelity vs Lindblad ({MODE})")

# Add grid and legend for better readability
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()

# Automatically save the figure 
filename_avg = f"Avg_Fidelity_Evolution_mode_{MODE}"
save_fig(fig01, filename_avg)

plt.show()

In [ ]:
# ===============================
# PLOT HEATMAP FIDELITY COMPLETE
# ===============================

plt.close('all')

# 1. Prepare the Data
# Transpose the matrix to get shape (N_traj, n_times)
fidelity_array = all_fidelity_matrix.T

n_traj, n_times = fidelity_array.shape

# 2. Heatmap Bin Parameters
n_bins = 150 
fidelity_bins = np.linspace(0.0, 1.0, n_bins + 1)
heatmap_complete = np.zeros((n_bins, n_times))

# Compute the histogram for each time step
for t in range(n_times):
    counts, _ = np.histogram(fidelity_array[:, t], bins=fidelity_bins)
    heatmap_complete[:, t] = counts

# 3. Create the Figure
# Explicitly set a wider figsize for the heatmap
fig, ax = plt.subplots(figsize=(12, 5))

# Mask zero counts to make the background transparent/white instead of dark
heatmap_masked_complete = np.ma.masked_where(heatmap_complete == 0, heatmap_complete) 

# 4. Plot the Heatmap
# Using the physical 'times' array for the x-axis extent to match other plots
im = ax.imshow(
    heatmap_masked_complete,
    aspect='auto',
    origin='lower',
    extent=[times[0], times[-1], 0.0, 1.0],  # [x_min, x_max, y_min, y_max]
    cmap='viridis',
    interpolation='nearest',
    vmin=1, 
    vmax=np.max(heatmap_complete) * 0.8 # Saturate slightly to highlight lower densities
)

# 5. Formatting
# Add colorbar 
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Number of Trajectories')

# Apply labels and dynamic title
ax.set_xlabel('Time')
ax.set_ylabel(r'Fidelity $\mathcal{F}$')
ax.set_title(f'Fidelity Distribution over Time ({MODE})')

# Automatically save the heatmap figure (uncomment when ready)
filename_heatmap = f"Heatmap_Fidelity_Complete_mode_{MODE}"
save_fig(fig, filename_heatmap)

plt.show()

In [ ]:
# =============================================
# Fidelity for a Single Trajectory vs Lindblad
# =============================================

plt.close('all')

# Choose which trajectory index to plot 
sample_idx = 0

# Extract the fidelity values for the chosen trajectory across all time steps
# We use the all_fidelity_matrix computed previously with Numba
single_traj_fidelity = all_fidelity_matrix[:, sample_idx]

# Create figure and axis using global aesthetic settings
fig, ax = plt.subplots()

# Plot the single trajectory
ax.plot(
    times, 
    single_traj_fidelity, 
    label=f'Single Traj (idx = {sample_idx})', 
    color='dodgerblue', 
    linewidth=1.5, 
    alpha=0.9
)

# Set labels and title
ax.set_xlabel("Time steps")
ax.set_ylabel(r"Fidelity $\mathcal{F}(\rho_{traj}^{(i)}, \rho_L)$")
ax.set_title(f"Single Trajectory Fidelity vs Lindblad | mode_{MODE}")
ax.legend()

# Automatically save the figure dynamically
filename_single = f"Single_Traj_Fidelity_idx{sample_idx}_mode_{MODE}"
#save_fig(fig, filename_single)

plt.show()

In [ ]:
# =================================================================
# Single Trajectory and Avg Fidelity over Trajectories vs Lindblad
# =================================================================

plt.close('all')

# Average fifelity across all trajectories for each time step
avg_fidelity_over_traj = np.mean(all_fidelity_matrix, axis=1)

# Choose which trajectory index to plot 
sample_idx = 0

# Extract the fidelity values for the chosen trajectory across all time steps
# We use the all_fidelity_matrix computed previously with Numba
single_traj_fidelity = all_fidelity_matrix[:, sample_idx]

# Create figure and axis using global aesthetic settings
fig, ax = plt.subplots()

# Plot the single trajectory
ax.plot(
    times, 
    avg_fidelity_over_traj, 
    label=f'Avg Fidelity Over Trajectories', 
    color='blue', 
    linewidth=1.5, 
    alpha=0.9
)

# Plot the single trajectory
ax.plot(
    times, 
    single_traj_fidelity, 
    label=f'Single Traj (idx = {sample_idx})', 
    color='red', 
    linewidth=1.5, 
    alpha=0.9
)

# Set labels and title
ax.set_xlabel("Time steps")
ax.set_ylabel(r"Fidelity Over Trajectories")
ax.set_title(f"Single Traj and Avg Fidelity Over Trajectories vs Lindblad | mode_{MODE}")
ax.legend()

# Automatically save the figure dynamically
filename_single = f"Single Traj and Avg Fidelity Over Trajectories vs Lindblad | mode_{MODE}"
#save_fig(fig, filename_single)

plt.show()

## Trace Distance calculation

In [ ]:
# ============================================
# PLOT HEATMAP: COMPLETE TRACE DISTANCE
# ============================================

plt.close('all')

# Transpose the array to get shape (n_trajectories, n_timesteps)
td_array = np.array(all_trace_dist_matrix).T

n_traj, n_times = td_array.shape

# Heatmap bin parameters for Trace Distance (values range from 0 to 1)
n_bins = 150 
td_bins = np.linspace(0.0, 1.0, n_bins + 1)
heatmap_td = np.zeros((n_bins, n_times))

# Compute the histogram for each time step
for t in range(n_times):
    counts, _ = np.histogram(td_array[:, t], bins=td_bins)
    # Normalize by the total number of trajectories to get a probability fraction
    heatmap_td[:, t] = counts / n_traj

# Create the figure with a wider format for better time resolution
fig, ax = plt.subplots(figsize=(12, 6))

# Mask exactly zero values so they appear as background and don't break the LogNorm
heatmap_masked_td = np.ma.masked_where(heatmap_td == 0, heatmap_td) 

# The minimum non-zero fraction possible is 1 / n_traj
vmin_val = 1.0 / n_traj  

# Plot the heatmap using Logarithmic Normalization
im = ax.imshow(
    heatmap_masked_td,
    aspect='auto',
    origin='lower',
    extent=[times[0], times[-1], 0.0, 1.0],
    cmap='magma', 
    norm=LogNorm(vmin=vmin_val, vmax=1.0),
    interpolation='nearest'
)

# Add colorbar with explicit label
cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label('Fraction of trajectories (Log Scale)')

# Apply labels and dynamic title
ax.set_xlabel('Time')
ax.set_ylabel(r'Trace Distance $\mathcal{T}(\rho_{traj}, \rho_L)$')
ax.set_title(f'Trace Distance Probability Density over Time | mode_{MODE}')

# Automatically save the heatmap figure (uncomment when needed)
filename_heatmap_td = f"Heatmap_Trace_Distance_Complete_mode_{MODE} "
save_fig(fig, filename_heatmap_td)

plt.show()

In [ ]:
# ==========================================================
# Trace Distance of the Average Trajectory vs Lindblad
# ==========================================================

plt.close('all')

# List of trajectories to average over
N_list = [100, 1000, 10000]

# Create the figure and axis
fig, ax = plt.subplots(figsize=(8, 5))

for N in N_list:
    # 1. Fast calculation of average populations and coherences for N trajectories
    pop_00_m = np.mean(pop_00[:, :N], axis=1)
    pop_11_m = np.mean(pop_11[:, :N], axis=1)
    coh_10_m = np.mean(coh_10[:, :N], axis=1)
    coh_01_m = np.mean(coh_01[:, :N], axis=1)
    
    # 2. Use the optimized Numba function to get the Trace Distance directly
    # We use '_' for the first output (Fidelity) since we only need the Trace Distance here
    _, td_mean_array = compute_metrics_over_time(
        pop_00_m, pop_11_m, coh_10_m, coh_01_m, rho_lindblad
    )
    
    # 3. Dynamic styling: the curve for the maximum N will be the most prominent
    alpha_val = 0.6 if N < max(N_list) else 1.0
    lw = 1.5 if N < max(N_list) else 2.5
    
    # Plot the curve using physical times
    ax.plot(times, td_mean_array, label=f'N_traj = {N}', alpha=alpha_val, linewidth=lw)
    
# Apply labels and title
ax.set_xlabel("Time")
ax.set_ylabel(r"Trace Distance $\mathcal{D}(\langle\rho\rangle_t, \rho_L)$")
ax.set_title(f"Evolution of Avg Trace Distance vs Lindblad ({MODE})")

# Add grid and legend
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)

# Optional: zoom in on the y-axis to see the noise near zero
# ax.set_ylim(-0.005, 0.1)

# Automatically save the figure (uncomment when ready)
filename_td = f"Avg_Trace_Distance_Evolution_mode_{MODE}"
save_fig(fig, filename_td)

plt.show()

## Variance Analysis

### Fidelity

In [ ]:
# =====================
# Variance Calculation
# =====================
variance_fidelity= np.var(all_fidelity_matrix, axis=1)

plt.close('all')

fig, ax = plt.subplots()
ax.plot(times, variance_fidelity, label='Variance of 1 - Fidelity', color='blue')   
ax.set_xlabel("Time steps")
ax.set_ylabel("Variance of 1 -Fidelity")
ax.set_title(f"Variance of 1 -Fidelity across Trajectories | mode_{MODE}")
ax.legend() 

# Automatically save figure 
filename= f"Fidelity_Variance_over_Trajectories_mode_{MODE}"
save_fig(fig, filename)

plt.show()

### Trace Distance

In [ ]:
# =====================
# Variance Calculation
# =====================
variance_trace_distance= np.var(all_trace_dist_matrix, axis=1)

plt.close('all')

fig, ax = plt.subplots()
ax.plot(times, variance_trace_distance, label='Variance of Trace Distance', color='red')   
ax.set_xlabel("Time steps")
ax.set_ylabel("Variance of Trace Distance")
ax.set_title(f"Variance of Trace Distance across Trajectories | mode_{MODE}")
ax.legend() 

# Automatically save figure 
filename= f"Trace_Distance_Variance_over_Trajectories_mode_{MODE}"
save_fig(fig, filename)

plt.show()